# EE4228 — Integrated Face Detection & Recognition System
**Live Demo Notebook**

### Workflow
| Step | Button | What happens |
|------|--------|--------------|
| 1 | Enter name → Start Capture | Opens webcam, press `S` to save augmented photos |
| 2 | Train Model | Detects faces in saved images, extracts embeddings, saves DB |
| 3 | Live Recognition | Opens webcam with live bounding boxes + identity labels |

**Matcher** (dropdown in UI): `cosine` · `knn` · `svm`  
**Threshold slider**: adjust confidence cutoff live

## 0. Install dependencies (run once)

In [1]:
import subprocess, sys
pkgs = ['facenet-pytorch', 'opencv-python', 'torch', 'torchvision',
        'Pillow', 'numpy', 'scikit-learn']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('✓ Dependencies ready')

✓ Dependencies ready


In [7]:
import cv2

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
ret, frame = cap.read()
print("ret:", ret)
print("frame shape:", frame.shape if ret else "no frame")
cap.release()

ret: True
frame shape: (480, 640, 3)


## 1. Run the integrated app

In [8]:
import cv2
import os
import pickle
import numpy as np
import tkinter as tk
from tkinter import ttk, messagebox
from PIL import Image, ImageTk
from facenet_pytorch import MTCNN, InceptionResnetV1
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import normalize
import threading
import time

# ── Configuration ─────────────────────────────────────────────────────────────
DATABASE_DIR  = 'database'
DATABASE_FILE = 'database/face_embeddings.pkl'
NUM_CAPTURES  = 5      # press S this many times per person (×4 augments = 20 images)
IMG_SIZE      = 160


# ── Augmentation ──────────────────────────────────────────────────────────────
def apply_augmentations(frame: np.ndarray) -> dict:
    """
    Generate 4 variants of a captured frame to expand the dataset.
    Each press of S saves 4 images → 5 presses = 20 images per person.
    """
    h, w = frame.shape[:2]
    # Rotate ±10 degrees
    M_l = cv2.getRotationMatrix2D((w/2, h/2), 10, 1.0)
    M_r = cv2.getRotationMatrix2D((w/2, h/2), -10, 1.0)
    return {
        'raw':    frame,
        'flip':   cv2.flip(frame, 1),
        'bright': cv2.convertScaleAbs(frame, alpha=1.3, beta=25),
        'dark':   cv2.convertScaleAbs(frame, alpha=0.75, beta=-25),
        'rot_l':  cv2.warpAffine(frame, M_l, (w, h), borderMode=cv2.BORDER_REFLECT_101),
        'rot_r':  cv2.warpAffine(frame, M_r, (w, h), borderMode=cv2.BORDER_REFLECT_101),
    }


# ── Embedding matcher ─────────────────────────────────────────────────────────
class EmbeddingMatcher:
    """
    Handles cosine similarity, KNN, and SVM matching against the
    stored face embedding database.
    """
    def __init__(self, strategy='cosine', threshold=0.55, k=3):
        self.strategy  = strategy
        self.threshold = threshold
        self.k = k
        self._clf = None
        self._centroids = {}

    def fit(self, database: dict):
        """Fit matcher from {name: [emb, ...]} database."""
        if not database:
            self._centroids = {}
            self._clf = None
            return

        # Always build centroids (cosine fallback)
        self._centroids = {
            name: normalize(np.mean(embs, axis=0, keepdims=True))[0]
            for name, embs in database.items()
        }

        if self.strategy in ('knn', 'svm') and len(database) >= 2:
            X, y = [], []
            for name, embs in database.items():
                X.extend(embs)
                y.extend([name] * len(embs))
            X = np.array(X)
            if self.strategy == 'knn':
                self._clf = KNeighborsClassifier(
                    n_neighbors=min(self.k, len(X)),
                    metric='cosine', algorithm='brute')
                self._clf.fit(X, y)
            else:  # svm
                base = LinearSVC(max_iter=5000, C=1.0)
                self._clf = CalibratedClassifierCV(base, cv=min(3, len(database)))
                self._clf.fit(X, y)

    def match(self, emb: np.ndarray) -> tuple:
        """Return (name, confidence). name='Unknown' if below threshold."""
        if not self._centroids:
            return 'Unknown', 0.0

        if self.strategy == 'cosine' or self._clf is None:
            best_name, best_score = 'Unknown', 0.0
            for name, centroid in self._centroids.items():
                score = float(np.dot(emb, centroid))
                if score > best_score:
                    best_score, best_name = score, name
            if best_score < self.threshold:
                return 'Unknown', best_score
            return best_name, best_score
        else:
            proba  = self._clf.predict_proba(emb.reshape(1, -1))[0]
            idx    = int(np.argmax(proba))
            score  = float(proba[idx])
            if score < 0.40:
                return 'Unknown', score
            return self._clf.classes_[idx], score


# ── Main application ──────────────────────────────────────────────────────────
class IntegratedFaceApp:
    """
    Integrated Face Detection and Recognition System.

    Architecture (two threads):
      Main thread  → Tkinter event loop
      update_video → called via window.after() — camera + inference in main thread
                     (Tkinter is single-threaded; training uses a daemon thread)
    """

    # Colour palette for bounding boxes (BGR)
    _COLORS = [
        (0, 200, 80), (255, 160, 0), (0, 180, 255),
        (200, 0, 255), (255, 80, 160), (0, 220, 180)
    ]
    _UNKNOWN_COLOR = (60, 60, 220)

    def __init__(self, window):
        self.window = window
        self.window.title('EE4228: Integrated Face System')
        self.window.geometry('1150x700')
        self.window.configure(bg='#2c3e50')

        # ── Models ────────────────────────────────────────────────────────────
        self.mtcnn  = MTCNN(image_size=IMG_SIZE, margin=20,
                            keep_all=True, min_face_size=40)
        self.resnet = InceptionResnetV1(pretrained='vggface2').eval()

        # ── State ─────────────────────────────────────────────────────────────
        self.cap           = None
        self.current_mode  = 'Idle'
        self.raw_database  = {}     # {name: [emb, emb, ...]}
        self.capture_count = 0
        self.target_name   = ''
        self._id_colors    = {}
        self._color_idx    = 0
        self._fps_ema      = 0.0
        self._prev_t       = time.perf_counter()
        self._last_frame   = None

        # ── Matcher ───────────────────────────────────────────────────────────
        self._matcher_var   = tk.StringVar(value='cosine')
        self._threshold_var = tk.DoubleVar(value=0.55)
        self.matcher = EmbeddingMatcher(strategy='cosine', threshold=0.55)

        self._load_database()
        self._build_ui()

    # ── Database ──────────────────────────────────────────────────────────────

    def _load_database(self):
        if os.path.exists(DATABASE_FILE):
            with open(DATABASE_FILE, 'rb') as f:
                data = pickle.load(f)
            # Support both old (centroids-only) and new (raw embeddings) format
            if data and isinstance(list(data.values())[0], list):
                self.raw_database = data
            else:
                # Legacy: convert centroid-only DB to single-embedding lists
                self.raw_database = {k: [v] for k, v in data.items()}
            self._refit_matcher()
            self._set_status(f'Database loaded: {len(self.raw_database)} identities')

    def _save_database(self):
        os.makedirs(DATABASE_DIR, exist_ok=True)
        with open(DATABASE_FILE, 'wb') as f:
            pickle.dump(self.raw_database, f)

    def _refit_matcher(self):
        strategy  = self._matcher_var.get()
        threshold = self._threshold_var.get()
        self.matcher = EmbeddingMatcher(strategy=strategy, threshold=threshold)
        self.matcher.fit(self.raw_database)

    # ── UI ────────────────────────────────────────────────────────────────────

    def _build_ui(self):
        # Sidebar
        sidebar = tk.Frame(self.window, width=290, bg='#34495e')
        sidebar.pack(side=tk.LEFT, fill=tk.Y)
        sidebar.pack_propagate(False)

        def lbl(text, pady=(10,0)):
            tk.Label(sidebar, text=text, fg='#bdc3c7', bg='#34495e',
                     font=('Arial', 10)).pack(pady=pady)

        def btn(text, cmd, color='#2980b9'):
            tk.Button(sidebar, text=text, command=cmd, bg=color,
                      fg='white', font=('Arial', 10, 'bold'),
                      width=24, pady=4).pack(pady=4, padx=10)

        lbl('── CAPTURE ──', (20, 2))
        lbl('Step 1: Enter name')
        self.name_entry = tk.Entry(sidebar, font=('Arial', 11), width=22)
        self.name_entry.insert(0, 'Enter_Name')
        self.name_entry.pack(pady=4)

        btn('2. Start Photo Capture', self._setup_capture, '#2980b9')
        btn('3. Train / Update Model', self._start_training, '#f39c12')
        btn('4. Live Recognition', self._setup_recognition, '#27ae60')
        btn('Stop Camera', self._stop_camera, '#c0392b')

        lbl('── MATCHER ──', (16, 2))
        for m in ('cosine', 'knn', 'svm'):
            tk.Radiobutton(
                sidebar, text=m.upper(), variable=self._matcher_var,
                value=m, command=self._refit_matcher,
                bg='#34495e', fg='white', selectcolor='#2c3e50',
                font=('Arial', 10)
            ).pack(anchor=tk.W, padx=20)

        lbl('Threshold')
        self._thr_label = tk.Label(sidebar, text='0.55',
                                   fg='white', bg='#34495e', font=('Arial', 10, 'bold'))
        self._thr_label.pack()
        tk.Scale(
            sidebar, from_=0.10, to=0.95, resolution=0.01,
            orient=tk.HORIZONTAL, variable=self._threshold_var,
            command=self._on_threshold, length=240,
            bg='#34495e', fg='white', troughcolor='#2c3e50',
            highlightthickness=0, showvalue=False
        ).pack(padx=10)

        lbl('── DATABASE ──', (16, 2))
        self._db_label = tk.Label(sidebar, text='No database loaded',
                                  fg='#95a5a6', bg='#34495e', font=('Arial', 9),
                                  wraplength=250, justify=tk.LEFT)
        self._db_label.pack(padx=10, pady=4)

        # Status bar at bottom of sidebar
        self._status_var = tk.StringVar(value='Ready')
        tk.Label(sidebar, textvariable=self._status_var, fg='#f1c40f',
                 bg='#34495e', font=('Arial', 9), wraplength=270,
                 justify=tk.LEFT).pack(side=tk.BOTTOM, pady=8, padx=8)

        # FPS label
        self._fps_var = tk.StringVar(value='FPS: —')
        tk.Label(sidebar, textvariable=self._fps_var, fg='#2ecc71',
                 bg='#34495e', font=('Arial', 9, 'bold')
                 ).pack(side=tk.BOTTOM, pady=2)

        # Video canvas
        self.video_label = tk.Label(self.window, bg='black')
        self.video_label.pack(side=tk.RIGHT, expand=True, fill=tk.BOTH,
                              padx=6, pady=6)

        self._update_db_label()

    def _on_threshold(self, _=None):
        val = round(self._threshold_var.get(), 2)
        self._thr_label.config(text=f'{val:.2f}')
        self.matcher.threshold = val

    def _set_status(self, msg):
        self.window.after(0, lambda: self._status_var.set(msg))

    def _update_db_label(self):
        if self.raw_database:
            lines = [f'  {n}: {len(e)} embeddings'
                     for n, e in sorted(self.raw_database.items())]
            self._db_label.config(
                text='Identities:\n' + '\n'.join(lines),
                fg='#2ecc71'
            )
        else:
            self._db_label.config(text='No database loaded', fg='#95a5a6')

    # ── Colour helpers ────────────────────────────────────────────────────────

    def _get_color(self, name):
        if name == 'Unknown':
            return self._UNKNOWN_COLOR
        if name not in self._id_colors:
            self._id_colors[name] = self._COLORS[self._color_idx % len(self._COLORS)]
            self._color_idx += 1
        return self._id_colors[name]

    # ── Camera control ────────────────────────────────────────────────────────

    def _start_webcam(self):
        if self.cap is None or not self.cap.isOpened():
            self.cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
            self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
            self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
        self._update_video()

    def _stop_camera(self):
        self.current_mode = 'Idle'
        if self.cap:
            self.cap.release()
            self.cap = None
        self.video_label.configure(image='', bg='black')
        self._set_status('Camera stopped')
        self._fps_var.set('FPS: —')

    # ── Capture mode ──────────────────────────────────────────────────────────

    def _setup_capture(self):
        label = self.name_entry.get().strip().lower().replace(' ', '_')
        if not label or label == 'enter_name':
            messagebox.showwarning('Error', 'Please enter a valid name first.')
            return
        self.target_name   = label
        self.capture_count = 0
        self.current_mode  = 'Capture'
        self._set_status(f'Capture mode — press S. Target: {label}')
        self._start_webcam()

    def _handle_keypress(self, event):
        if self.current_mode == 'Capture' and event.char.lower() == 's':
            self._capture_one()

    def _capture_one(self):
        if self.cap is None or self.capture_count >= NUM_CAPTURES:
            return
        ret, frame = self.cap.read()
        if not ret:
            return
        frame = cv2.flip(frame, 1)

        save_path = os.path.join(DATABASE_DIR, self.target_name)
        os.makedirs(save_path, exist_ok=True)

        variants = apply_augmentations(frame)
        for v_name, v_frame in variants.items():
            filename = f'{self.target_name}_{self.capture_count:03d}_{v_name}.jpg'
            cv2.imwrite(os.path.join(save_path, filename), v_frame)

        self.capture_count += 1
        self._set_status(
            f'Captured {self.capture_count}/{NUM_CAPTURES} '
            f'({self.capture_count * len(variants)} images saved)'
        )

        if self.capture_count >= NUM_CAPTURES:
            total = NUM_CAPTURES * len(variants)
            messagebox.showinfo(
                'Capture complete',
                f'Saved {total} images for "{self.target_name}".\n'
                f'Click "Train / Update Model" to register this identity.'
            )
            self.current_mode = 'Idle'

    # ── Training ──────────────────────────────────────────────────────────────

    def _start_training(self):
        self._set_status('Training... please wait')
        threading.Thread(target=self._train_logic, daemon=True).start()

    def _train_logic(self):
        """
        For each identity folder: detect face crops with MTCNN,
        extract 512-d FaceNet embeddings, store all embeddings per identity.
        Saves raw embeddings (not just centroids) so all 3 matchers work.
        """
        if not os.path.exists(DATABASE_DIR):
            os.makedirs(DATABASE_DIR)

        new_db = {}
        identities = [
            d for d in os.listdir(DATABASE_DIR)
            if os.path.isdir(os.path.join(DATABASE_DIR, d))
        ]

        for person in identities:
            p_path = os.path.join(DATABASE_DIR, person)
            embs   = []

            # Only process image files — skip .ipynb_checkpoints etc.
            img_files = [
                f for f in os.listdir(p_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ]

            for img_name in img_files:
                try:
                    img  = Image.open(os.path.join(p_path, img_name)).convert('RGB')
                    crop = self.mtcnn(img)   # returns tensor(s) or None

                    if crop is None:
                        continue

                    # MTCNN can return [N,3,H,W] or [3,H,W] depending on keep_all
                    if crop.dim() == 3:
                        crop = crop.unsqueeze(0)   # add batch dim

                    # Take only the first (largest) detected face
                    face_tensor = crop[0].unsqueeze(0)

                    import torch
                    with torch.no_grad():
                        emb = self.resnet(face_tensor).detach().numpy()[0]

                    # L2-normalise
                    emb = emb / (np.linalg.norm(emb) + 1e-10)
                    embs.append(emb)

                except Exception as e:
                    # Skip unreadable files silently
                    pass

            if embs:
                new_db[person] = embs
                self._set_status(f'Trained: {person} ({len(embs)} embeddings)')

        self.raw_database = new_db
        self._save_database()
        self._refit_matcher()

        self.window.after(0, self._update_db_label)
        total_embs = sum(len(v) for v in new_db.values())
        messagebox.showinfo(
            'Training complete',
            f'Trained {len(new_db)} identities / {total_embs} embeddings.\n'
            f'Matcher: {self._matcher_var.get().upper()}'
        )
        self._set_status(f'Ready — {len(new_db)} identities in DB')

    # ── Recognition mode ──────────────────────────────────────────────────────

    def _setup_recognition(self):
        if not self.raw_database:
            messagebox.showwarning('Warning', 'Database is empty — train the model first.')
            return
        self.current_mode = 'Recognition'
        self._set_status(f'Live recognition — matcher: {self._matcher_var.get().upper()}')
        self._start_webcam()

    # ── Video loop ────────────────────────────────────────────────────────────

    def _update_video(self):
        if self.cap is None or not self.cap.isOpened():
            return

        ret, frame = self.cap.read()
        if not ret:
            self.window.after(30, self._update_video)
            return

        frame = cv2.flip(frame, 1)   # mirror view
        self._last_frame = frame.copy()

        # ── Recognition inference ────────────────────────────────────────────
        if self.current_mode == 'Recognition':
            img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            boxes, probs = self.mtcnn.detect(img_pil)

            if boxes is not None:
                faces = self.mtcnn.extract(img_pil, boxes, save_path=None)

                if faces is not None:
                    import torch
                    with torch.no_grad():
                        embeddings = self.resnet(faces).detach().numpy()

                    for i, raw_emb in enumerate(embeddings):
                        # L2-normalise
                        emb = raw_emb / (np.linalg.norm(raw_emb) + 1e-10)

                        # ── Identity lookup ──────────────────────────────────
                        name, conf = self.matcher.match(emb)

                        # ── Draw bounding box ────────────────────────────────
                        box   = boxes[i].astype(int)
                        x1, y1, x2, y2 = box[0], box[1], box[2], box[3]
                        h, w  = frame.shape[:2]
                        x1, y1 = max(0, x1), max(0, y1)
                        x2, y2 = min(w-1, x2), min(h-1, y2)
                        color  = self._get_color(name)

                        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

                        label = f'{name}  {conf:.2f}'
                        (tw, th), bl = cv2.getTextSize(
                            label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
                        ly = max(y1 - 6, th + 6)
                        cv2.rectangle(frame,
                                      (x1, ly - th - 6), (x1 + tw + 8, ly + bl),
                                      color, -1)
                        cv2.putText(frame, label, (x1 + 4, ly - 2),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                                    (255, 255, 255), 1, cv2.LINE_AA)

            # HUD
            n_faces = len(boxes) if boxes is not None else 0
            hud = (f'Matcher: {self._matcher_var.get().upper()}  '
                   f'Threshold: {self._threshold_var.get():.2f}  '
                   f'Faces: {n_faces}  '
                   f'DB: {len(self.raw_database)} identities')
            cv2.putText(frame, hud, (8, frame.shape[0] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                        (200, 200, 200), 1, cv2.LINE_AA)

        # ── Capture guide ─────────────────────────────────────────────────────
        elif self.current_mode == 'Capture':
            h, w = frame.shape[:2]
            cx, cy = w // 2, h // 2
            cv2.rectangle(frame, (cx-160, cy-180), (cx+160, cy+180), (0, 255, 0), 2)
            cv2.putText(
                frame,
                f"Press 'S' to capture  {self.capture_count}/{NUM_CAPTURES}  "
                f"Name: {self.target_name}",
                (12, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
            )

        # ── FPS ───────────────────────────────────────────────────────────────
        now = time.perf_counter()
        inst = 1.0 / max(now - self._prev_t, 1e-6)
        self._prev_t = now
        self._fps_ema = 0.1 * inst + 0.9 * self._fps_ema
        self._fps_var.set(f'FPS: {self._fps_ema:.1f}')

        # ── Display ───────────────────────────────────────────────────────────
        img_tk = ImageTk.PhotoImage(
            Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        )
        self.video_label.configure(image=img_tk)
        self.video_label.imgtk = img_tk

        self.window.after(10, self._update_video)

    # ── Cleanup ───────────────────────────────────────────────────────────────

    def _on_close(self):
        self._stop_camera()
        self.window.destroy()


# ── Launch ────────────────────────────────────────────────────────────────────
root = tk.Tk()
app  = IntegratedFaceApp(root)
root.bind('<Key>', app._handle_keypress)
root.protocol('WM_DELETE_WINDOW', app._on_close)
root.mainloop()

In [ ]:
# ── (Optional) Quick DB inspection ───────────────────────────────────────────
import pickle, os

DB_FILE = 'database/face_embeddings.pkl'
if os.path.exists(DB_FILE):
    with open(DB_FILE, 'rb') as f:
        db = pickle.load(f)
    print(f'Identities in database: {len(db)}')
    for name, embs in sorted(db.items()):
        n = len(embs) if isinstance(embs, list) else 1
        print(f'  {name:<20} {n:>4} embeddings')
else:
    print('No database file found — run training first.')